# 02 — PyTorch CNN starter
Setup, the device check, and data loading work now. DataLoaders, architecture, training, and evaluation remain TODOs for the CNN team.

## 1. Setup

### Environment and dataset configuration
**The one dataset path you may need to change is `DRIVE_DATA_ROOT` below.** It is an example, not a known shared-folder location. Point to the folder directly containing `Train.csv`, `Test.csv`, `Train/`, and `Test/`. If the folder is under "Shared with me", add a shortcut under My Drive first.

Colab defaults to Drive; local runs default to `Dataset/raw/`. Set `USE_DRIVE = False` to use repository data in Colab. Run cells in order. No API keys are needed.

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB
# CHANGE THIS ONE DATASET PATH to your own shared Drive folder/shortcut:
DRIVE_DATA_ROOT = "/content/drive/MyDrive/CP467_Traffic_Signs/data/raw"
print("Running in Colab:", IN_COLAB)
print("Python:", sys.version)

### Mount Drive and locate project code
Opening a GitHub notebook in Colab does not clone its Python files. This cell finds a local project checkout or clones it once in Colab. For a code checkout stored on Drive, change the working directory to that checkout before this cell (see README).

In [ ]:
import os
import subprocess

if USE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None and IN_COLAB:
    PROJECT_ROOT = Path("/content/CP467GroupProject")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Davy-Chu/CP467GroupProject.git", str(PROJECT_ROOT)],
            check=True,
        )
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Open from the project checkout or change directory to its root first.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

### Dependencies
Colab often includes these libraries. Leave installation off first; if an import fails, set `INSTALL_DEPENDENCIES = True`, run this cell, restart the runtime if requested, then rerun from the top with installation off. Avoid replacing a working Colab GPU environment unnecessarily.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import skimage
import sklearn
import torch
import torchvision
import seaborn

for library in (cv2, np, pd, matplotlib, skimage, sklearn, torch, torchvision, seaborn):
    print(f"{library.__name__}: {library.__version__}")

In [ ]:
from src.config import (
    NUM_CLASSES, IMAGE_SIZE, RANDOM_SEED, REPO_DATA_ROOT,
    RESULTS_DIR, MODELS_DIR, CONDITIONS, DEGRADATION_PARAMS,
    get_data_root, seed_everything,
)
from src.data_utils import (
    validate_dataset, load_metadata, load_image,
    make_train_validation_split, split_fingerprint,
)
from src.degradations import (
    apply_condition, reduce_brightness, add_noise, apply_motion_blur,
    equalize_histogram, gaussian_filter,
)

seed_everything()
DATA_ROOT = get_data_root(USE_DRIVE, DRIVE_DATA_ROOT)
print("Selected data root:", DATA_ROOT)
print("Repository fallback:", REPO_DATA_ROOT)
print("Shared conditions:", DEGRADATION_PARAMS)

## 2. GPU / Device Check
In Colab choose **Runtime → Change runtime type → GPU** before running. CPU is enough for these setup checks.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("CPU selected. Enable a Colab GPU before CNN training if available.")

## 3. Load Dataset

Validation requires all 43 classes, the full 39,209/12,630 labeled image counts, and every referenced file. It fails if anything required is missing. The default decodes one image per class in each split; use `check_images="all"` for a slower full corruption scan on Drive. Do not disable the count checks to work around an unfinished upload.

In [ ]:
# This must pass before either team begins model development.
training_metadata, test_metadata = validate_dataset(DATA_ROOT, check_images="sample")
train_metadata, validation_metadata = make_train_validation_split(training_metadata)
SPLIT_ID = split_fingerprint(train_metadata, validation_metadata)
print(f"Training subset: {len(train_metadata):,}")
print(f"Validation subset: {len(validation_metadata):,}")
print(f"Official test (final evaluation only): {len(test_metadata):,}")
print("Split ID (must match across teammates):", SPLIT_ID)

## 4. DataLoader Setup
Apply `apply_condition` to the common resized RGB image before converting to tensors. Reuse optional shared preprocessing helpers; never augment validation/test randomly.

In [ ]:
# TODO: Implement a Dataset using the shared metadata and load_image.
# Convert RGB HWC uint8 to CHW floating tensors and document normalization.
# Shuffle training only; preserve validation/test identity through row.Path.
# Seed torch.Generator and worker RNGs if using multiple workers.

## 5. CNN Architecture

In [ ]:
# TODO: Implement and document a suitable torch.nn.Module with NUM_CLASSES outputs.
# Choose architecture settings during development, then move the model to device.

## 6. Training

In [ ]:
# TODO: Implement training using train_metadata only; log real losses and timing.
# Select loss/optimizer/settings during development; save temporary checkpoints
# in models/checkpoints/ (ignored by Git).

## 7. Validation

In [ ]:
# TODO: Evaluate without gradients in model.eval() on validation_metadata.
# Use validation for model selection and freeze choices before testing.

## 8. Clean Test Evaluation

In [ ]:
# TODO: Evaluate the frozen model on test_metadata with condition='clean'.
# Use accuracy and macro F1 with labels=list(range(NUM_CLASSES)), zero_division=0.
# Match the HOG+SVM evaluation definitions.

## 9. Degraded Test Evaluation

In [ ]:
# TODO: Loop over CONDITIONS with apply_condition(image, condition, image_id=row.Path).
# Reuse the same test rows and shared degradation settings as HOG+SVM.
# Evaluate optional preprocessing separately; never tune on the test set.

## 10. Timing

In [ ]:
# TODO: Use time.perf_counter. Synchronize CUDA before/after timed GPU work.
# Warm up first; report total training seconds and mean inference ms/image.
# Record hardware, batch size, and preprocessing inclusion to match HOG+SVM.

## 11. Confusion Matrix

In [ ]:
# TODO: Compute a 43x43 matrix using labels=list(range(NUM_CLASSES)).
# Save labeled figures under RESULTS_DIR / 'confusion_matrices'.

## 12. Save Model / Results
No model artifacts or experiment results are generated by this starter.

In [ ]:
# TODO: Save a reasonably sized final state_dict under MODELS_DIR / 'cnn.pt'.
# Append real metrics to RESULTS_DIR / 'metrics.csv' using the existing header.
# Record preprocessing, RANDOM_SEED, SPLIT_ID, library versions, and settings.
# Save plots under results/plots/; keep large artifacts in Drive.